# 05 Lemmatize

Reads `processed/04_abbreviations.csv` and reduces each word to its dictionary form with spaCy ("legs" becomes "leg", "walking" becomes "walk", "ran" becomes "run"). Each answer is lemmatized as a whole, so the tagger can use the neighbouring words to decide whether "saw" is a noun or a verb.

Punctuation is dropped here (the later steps work one word at a time). Numbers and words are kept.

Output: `processed/05_lemmatized.csv` with
- `answer_lemmatized`: the lemmas, in order, separated by spaces.
- `tokens`: `word|lemma|pos` for every word, separated by spaces, so the original word is still there for the final file.

In [1]:
import pandas as pd
import spacy
from collections import Counter

# the parser and entity recogniser are not needed
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
print(nlp.pipe_names)

['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


In [2]:
data = pd.read_csv("processed/04_abbreviations.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,465 rows


In [3]:
# nouns that are only used in the plural: the singular means something else, so keep them as written
KEEP_AS_WRITTEN = {
    "clothes", "pants", "glasses", "jeans", "shorts", "scissors", "stairs", "species", "politics",
    "mathematics", "athletics", "ethics", "physics", "economics", "goggles", "tights", "trousers",
    "pliers", "headquarters", "crossroads", "outskirts", "belongings", "savings", "earnings",
}

def lemma_of(t):
    return t.text.lower() if t.text.lower() in KEEP_AS_WRITTEN else t.lemma_.lower()

lemmatized, tokens = [], []
changed = Counter()

for doc in nlp.pipe(data["answer_abbreviations"], batch_size=1000):
    words = [t for t in doc if not (t.is_punct or t.is_space)]
    lemmatized.append(" ".join(lemma_of(t) for t in words))
    tokens.append(" ".join(f"{t.text}|{lemma_of(t)}|{t.pos_}" for t in words))
    for t in words:
        if lemma_of(t) != t.text:
            changed[(t.text, lemma_of(t))] += 1

data["answer_lemmatized"] = lemmatized
data["tokens"] = tokens
print(f"{sum(changed.values()):,} words changed to a different form")

286,727 words changed to a different form


The most common changes, to check them:

In [4]:
log = pd.DataFrame([(a, b, n) for (a, b), n in changed.items()], columns=["word", "lemma", "count"])
log = log.sort_values("count", ascending=False)
log.to_csv("processed/05_lemma_changes.csv", index=False)
log.head(40)

,word,lemma,count
18,is,be,34394
30,used,use,10980
134,has,have,6807
169,are,be,5251
131,things,thing,4162
92,made,make,3309
255,being,be,2203
377,makes,make,1758
44,animals,animal,1613
172,found,find,1592


In [5]:
pd.set_option("display.max_colwidth", 100)
sample = data.sample(8, random_state=3)[["answer_abbreviations", "answer_lemmatized"]]
sample

,answer_abbreviations,answer_lemmatized
9645,knight maiden dame,knight maiden dame
66506,"ground beef sandwich, ground sirloin burger, salisbury steak, ground chuck, chopped beefsteak, c...",ground beef sandwich ground sirloin burger salisbury steak ground chuck chop beefsteak cheeseburger
105563,"an animal noise, the noise an owl makes",an animal noise the noise an owl make
134000,"dismantle, get rid of such as cargo, remove, leave off, omit",dismantle get rid of such as cargo remove leave off omit
44608,"comes off of trees, grows leaves, can be large or small, can break off, can hang a swing on, to ...",come off of tree grow leaf can be large or small can break off can hang a swing on to climb on
9017,territory range bound,territory range bind
69270,"beautiful, relaxing, sun, the horizon",beautiful relaxing sun the horizon
97159,"something to cover and treat a wound, a type of sauce used over food such as salad, to clothe on...",something to cover and treat a wound a type of sauce use over food such as salad to clothe oneself


In [6]:
out = data[["row_id", "subject_id", "cue", "answer_lemmatized", "tokens"]]
out.to_csv("processed/05_lemmatized.csv", index=False)
print(f"Saved {len(out):,} rows to processed/05_lemmatized.csv")

Saved 185,465 rows to processed/05_lemmatized.csv
